In [1]:
# ============================================================
#   MNIST DIGIT CLASSIFICATION – MINI PROJECT
#   Course: Machine Learning
# ============================================================

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "seaborn", "-q"], check=False)

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("=" * 60)
print("  Loading Dataset...")
print("=" * 60)

try:
    from sklearn.datasets import fetch_openml
    mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
    X_full, y_full = mnist.data / 255.0, mnist.target.astype(int)
    img_shape, dataset_name = (28, 28), "MNIST (OpenML)"
    print("  Loaded MNIST from OpenML")
except Exception:
    try:
        import tensorflow as tf
        (Xtr, ytr), (Xte, yte) = tf.keras.datasets.mnist.load_data()
        X_full = np.concatenate([Xtr.reshape(-1,784), Xte.reshape(-1,784)]) / 255.0
        y_full = np.concatenate([ytr, yte]).astype(int)
        img_shape, dataset_name = (28, 28), "MNIST (Keras)"
        print("  Loaded MNIST from Keras")
    except Exception:
        from sklearn.datasets import load_digits
        digits = load_digits()
        X_full, y_full = digits.data / 16.0, digits.target.astype(int)
        img_shape, dataset_name = (8, 8), "Digits (sklearn - offline demo)"
        print("  Using sklearn Digits dataset (offline fallback)")

print(f"  Dataset: {dataset_name}")
print(f"  Samples: {X_full.shape[0]}, Features: {X_full.shape[1]}")

MAX_SAMPLES = min(3000, X_full.shape[0])  # Reduced for speed
X_s, y_s = X_full[:MAX_SAMPLES], y_full[:MAX_SAMPLES]
X_train, X_test, y_train, y_test = train_test_split(
    X_s, y_s, test_size=0.2, random_state=42, stratify=y_s)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"  Train: {len(X_train)}, Test: {len(X_test)}")

# Sample digits plot
fig, axes = plt.subplots(2, 10, figsize=(18, 4))
fig.suptitle(f"Sample Handwritten Digits – {dataset_name}", fontsize=13, fontweight="bold")
for digit in range(10):
    idxs = np.where(y_train == digit)[0]
    for row in range(2):
        axes[row, digit].imshow(X_train[idxs[row]].reshape(img_shape), cmap="gray")
        if row == 0: axes[row, digit].set_title(f"Digit {digit}", fontsize=8)
        axes[row, digit].axis("off")
plt.tight_layout()
plt.savefig("sample_digits.png", dpi=150, bbox_inches="tight")
plt.close()
print("  [Saved] sample_digits.png")

results = {}

def train_evaluate(name, model, scaled=True):
    Xtr = X_train_sc if scaled else X_train
    Xte = X_test_sc  if scaled else X_test
    print(f"\n{'='*60}\n  MODEL: {name}\n{'='*60}")
    print(f"  Training...  ", end="", flush=True)
    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    print("Done.")
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, digits=4)
    cm = confusion_matrix(y_test, y_pred)
    print(f"  Accuracy: {acc*100:.2f}%")
    print(f"\n  Classification Report:\n{report}")
    # Confusion matrix heatmap
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=range(10), yticklabels=range(10),
                linewidths=0.4, linecolor="lightgrey", ax=ax)
    ax.set_title(f"Confusion Matrix – {name}\nAccuracy: {acc*100:.2f}%",
                 fontsize=14, fontweight="bold")
    ax.set_xlabel("Predicted Label", fontsize=12)
    ax.set_ylabel("True Label", fontsize=12)
    plt.tight_layout()
    fname = f"cm_{name.lower().replace(' ','_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  [Saved] {fname}")
    results[name] = {"accuracy": acc, "cm": cm, "y_pred": y_pred}

train_evaluate("KNN",           KNeighborsClassifier(n_neighbors=3, algorithm="ball_tree", n_jobs=-1))
train_evaluate("Decision Tree", DecisionTreeClassifier(max_depth=20, random_state=42))
train_evaluate("Random Forest", RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1))

# Comparison bar chart
model_names = list(results.keys())
accuracies  = [results[m]["accuracy"]*100 for m in model_names]
colors = ["#4C72B0", "#DD8452", "#55A868"]
fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(model_names, accuracies, color=colors, edgecolor="black", width=0.45)
ax.set_ylim(0, 115)
ax.set_ylabel("Accuracy (%)", fontsize=13)
ax.set_title("Model Accuracy Comparison – MNIST Digit Classification",
             fontsize=13, fontweight="bold")
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5,
            f"{acc:.2f}%", ha="center", fontweight="bold", fontsize=12)
ax.axhline(y=max(accuracies), color="red", ls="--", alpha=0.5,
           label=f"Best: {max(accuracies):.2f}%")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n  [Saved] model_comparison.png")

print(f"\n{'='*60}\n  COMPARISON SUMMARY\n{'='*60}")
for name, acc in zip(model_names, accuracies):
    marker = "  ← BEST" if acc == max(accuracies) else ""
    print(f"  {name:<20}: {acc:.2f}%{marker}")

# Error analysis
print(f"\n{'='*60}\n  ERROR ANALYSIS\n{'='*60}")
confusion_explanation = {
    (4,9):"4 and 9 share a vertical stroke and closed loop",
    (9,4):"4 and 9 share a vertical stroke and closed loop",
    (3,8):"3 and 8 have similar curved shapes",
    (8,3):"3 and 8 have similar curved shapes",
    (5,6):"5 and 6 both have a top bar and a loop",
    (6,5):"5 and 6 both have a top bar and a loop",
    (7,1):"7 and 1 share a dominant vertical stroke",
    (1,7):"7 and 1 share a dominant vertical stroke",
    (2,7):"Cursive 2 can resemble a mirrored 7",
    (3,5):"Lower portions of 3 and 5 are visually similar",
}
for name in model_names:
    cm_nd = results[name]["cm"].copy()
    np.fill_diagonal(cm_nd, 0)
    flat_idx = np.argsort(cm_nd.ravel())[::-1][:5]
    rows, cols = np.unravel_index(flat_idx, cm_nd.shape)
    print(f"\n  [{name}] Top confused pairs:")
    for r, c in zip(rows, cols):
        reason = confusion_explanation.get((r,c), "Visual structural similarity")
        print(f"    Digit {r} → {c}  |  count: {cm_nd[r,c]:3d}  |  {reason}")

# Misclassified examples
rf_pred  = np.array(results["Random Forest"]["y_pred"])
y_test_a = np.array(y_test)
wrong    = np.where(rf_pred != y_test_a)[0]
n_show   = min(20, len(wrong))
fig, axes = plt.subplots(4, 5, figsize=(14, 11))
fig.suptitle("Misclassified Digits – Random Forest", fontsize=13, fontweight="bold")
for i, ax in enumerate(axes.flat):
    if i >= n_show: ax.axis("off"); continue
    idx = wrong[i]
    ax.imshow(X_test[idx].reshape(img_shape), cmap="gray")
    ax.set_title(f"True:{y_test_a[idx]} → Pred:{rf_pred[idx]}",
                 color="crimson", fontsize=9, fontweight="bold")
    ax.axis("off")
plt.tight_layout()
plt.savefig("misclassified.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n  [Saved] misclassified.png")
print(f"\n{'='*60}\n  ALL DONE – 6 PNG files generated.\n{'='*60}")


  Loading Dataset...
  Loaded MNIST from OpenML
  Dataset: MNIST (OpenML)
  Samples: 70000, Features: 784
  Train: 2400, Test: 600
  [Saved] sample_digits.png

  MODEL: KNN
  Training...  Done.
  Accuracy: 89.00%

  Classification Report:
              precision    recall  f1-score   support

           0     0.9194    1.0000    0.9580        57
           1     0.8375    0.9853    0.9054        68
           2     0.8929    0.8333    0.8621        60
           3     0.8571    0.9153    0.8852        59
           4     0.8852    0.8308    0.8571        65
           5     0.9362    0.8000    0.8627        55
           6     0.9828    0.9344    0.9580        61
           7     0.8788    0.8788    0.8788        66
           8     0.9362    0.8462    0.8889        52
           9     0.8167    0.8596    0.8376        57

    accuracy                         0.8900       600
   macro avg     0.8943    0.8884    0.8894       600
weighted avg     0.8928    0.8900    0.8895       600

  